In [1]:
%load_ext autoreload
%autoreload 2

# `Logit` on Orders - Logistic Regression (~1h)

## Select features

🎯 Haydi `wait_time` ve `delay_vs_expected` değişkenlerinin çok `iyi/kötü review`lar üzerindeki etkisini inceleyelim.

👉 `orders` training_set’imizi kullanarak iki adet `multivariate logistic regression` çalıştıracağız:
- `logit_one` → `dim_is_one_star` tahmini için  
- `logit_five` → `dim_is_five_star` tahmini için.

 

In [2]:
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

👉 Dataset’inizi import edin:

In [3]:
from olist.order import Order
orders = Order().get_training_data(with_distance_seller_customer=True)

👉 Kullanmak istediğiniz feature’ları bir listede seçin:

⚠️ Data leakage yaratmadığınızdan emin olun (yani target’tan türetilmiş feature’ları seçmeyin)

💡 `wait_time` ve `delay_vs_expected` değişkenlerinin etkisini anlayabilmek için diğer feature’ların etkisini kontrol etmemiz gerekir, bu yüzden listenize ilgili olabilecek tüm feature’ları dahil edin.

In [12]:
features = [
    'wait_time',
    'delay_vs_expected',
    'number_of_items',
    'number_of_sellers',
    'price',
    'freight_value',
    'distance_seller_customer'
]

🕵🏻 Feature’larınızın `multicollinearity` durumunu `VIF index` kullanarak kontrol edin.

* Çok yüksek olmamalıdır (tercihen < 10), böylece partial regression coefficient’larına ve ilgili `p-values` değerlerine güvenebiliriz.
* Verinizi standardize etmeyi unutmayın!
    * Bir `VIF Analysis`, bir feature’ın diğer feature’lara karşı regresyonunu yaparak hesaplanır...
    * Bu yüzden herhangi bir linear regression çalıştırmadan önce feature’ların `scale etkisini kaldırmak` ve eşit öneme sahip olmalarını sağlamak istersiniz!
    
    
📚 <a href="https://www.statisticshowto.com/variance-inflation-factor/">Statistics How To - Variance Inflation Factor</a>

📚  <a href="https://online.stat.psu.edu/stat462/node/180/">PennState - Detecting Multicollinearity Using Variance Inflation Factors</a>

⚖️ Standardize etme:

In [13]:
from sklearn.preprocessing import StandardScaler

X = orders[features]

scaler = StandardScaler()
X_standardized = scaler.fit_transform(X)

X_standardized_df = pd.DataFrame(X_standardized, columns=features)

print("Veri başarıyla standardize edildi!")
display(X_standardized_df.head())

Veri başarıyla standardize edildi!


,wait_time,delay_vs_expected,number_of_items,number_of_sellers,price,freight_value,distance_seller_customer
0,-0.431195,-0.161782,-0.264596,-0.112545,-0.513805,-0.652042,-0.979480
1,0.134174,-0.161782,-0.264596,-0.112545,-0.086641,0.000467,0.429745
2,-0.329909,-0.161782,-0.264596,-0.112545,0.111749,-0.164054,-0.145496
3,0.073540,-0.161782,-0.264596,-0.112545,-0.441528,0.206816,2.054632
4,-1.019540,-0.161782,-0.264596,-0.112545,-0.562391,-0.652042,-0.959120


👉 Olası multicollinearity durumlarını analiz etmek için VIF Analysis’inizi çalıştırın:

In [15]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [16]:
vif_data = pd.DataFrame()
vif_data["feature"] = features

vif_data["VIF"] = [variance_inflation_factor(X_standardized_df.values, i)
                   for i in range(len(features))]

display(vif_data.sort_values(by="VIF", ascending=False))

,feature,VIF
0,wait_time,2.624944
1,delay_vs_expected,2.213500
5,freight_value,1.673727
6,distance_seller_customer,1.442040
2,number_of_items,1.371316
4,price,1.208582
3,number_of_sellers,1.093349


## Logistic Regressions

👉 İki adet `Logistic Regression` modeli fit edin:
- `logit_one` → `dim_is_one_star` tahmini için
- `logit_five` → `dim_is_five_star` tahmini için.

`Logit 1️⃣`

In [17]:
logit_data = X_standardized_df.copy()
logit_data['dim_is_one_star'] = orders['dim_is_one_star'].values
logit_data['dim_is_five_star'] = orders['dim_is_five_star'].values

formula = " + ".join(features)

logit_one = smf.logit(formula=f"dim_is_one_star ~ {formula}", data=logit_data).fit()

Optimization terminated successfully.
         Current function value: 0.273582
         Iterations 7


`Logit 5️⃣`

In [19]:
logit_five = smf.logit(formula=f"dim_is_five_star ~ {formula}", data=logit_data).fit()

print("Logit 1 Summary:")
print(logit_one.summary())
print("\n" + "="*50 + "\n")
print("Logit 5 Summary:")
print(logit_five.summary())

Optimization terminated successfully.
         Current function value: 0.636830
         Iterations 7
Logit 1 Summary:
                           Logit Regression Results                           
Dep. Variable:        dim_is_one_star   No. Observations:                95872
Model:                          Logit   Df Residuals:                    95864
Method:                           MLE   Df Model:                            7
Date:                Sun, 19 Apr 2026   Pseudo R-squ.:                  0.1448
Time:                        17:18:38   Log-Likelihood:                -26229.
converged:                       True   LL-Null:                       -30669.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept                   -2.4676      0.013   -190.852      

💡 Şimdi bu iki logistic regression’ın sonuçlarını analiz etme zamanı:

- Partial coefficient’ları kendi kelimelerinizle yorumlayın.
- `p-values` kullanarak istatistiksel anlamlılıklarını kontrol edin.
- Coefficient önemleri açısından `logit_one` ve `logit_five` arasında herhangi bir fark görüyor musunuz?

In [22]:
# Aşağıdaki cümlelerden doğru olanları aşağıdaki listeye kaydedin.

a = "delay_vs_expected influences five_star ratings even more than one_star ratings"
b = "wait_time influences five_star ratings even more than one_star"

your_answer = ["delay_vs_expected influences five_star ratings even more than one_star ratings"]

🧪 __Kodunu Test Et__

In [23]:
from nbresult import ChallengeResult

result = ChallengeResult('logit',
    answers = your_answer
)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/hamza/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /mnt/c/Users/emmo/code/hamza-simsek/data-logit/tests
plugins: typeguard-4.4.2, anyio-4.8.0, dash-4.1.0
collecting ... collected 1 item

test_logit.py::TestLogit::test_question PASSED                           [100%]

============================== 1 passed in 0.13s ===============================


💯 You can commit your code:

git add tests/logit.pickle

git commit -m 'Completed logit step'

git push origin master



<details>
    <summary>- <i>Açıklamalar ve ileri seviye kavramlar</i> -</summary>


> _Diğer tüm şeyler sabitken, `delay factor`, 1-yıldız review alma ihtimalini etkilemesinden bile daha fazla, 5-yıldızdan mahrum kalma ihtimalini artırma eğilimindedir. Muhtemelen bunun sebebi, 1-yıldız review’ların bizzat çok kötü ürünleri hedeflemesi, kötü teslimatları değil._

❗️ Ancak tamamen titiz olmak için, **iki farklı modelin coefficient’larını karşılaştırırken daha dikkatli olmamız gerekir**, çünkü **benzer popülasyonlara dayanmayabilirler**!
    Burada 2 alt popülasyonumuz var: (1-yıldız verenler ve 5-yıldız verenler) ve bunlar doğaları gereği farklı davranış kalıpları sergileyebilirler. 5-yıldız vermeye daha meyilli “mutlu insanlar”ın, “gecikme” veya “fiyat” söz konusu olduğunda, 1-yıldızı “Lucky-Luke gibi ateşleyen” “huysuz insanlara” göre daha az hassas olmaları gayet mümkün...

</details>



🏁 Tebrikler!

💾 `logit.ipynb` notebook’unuzu commit ve push etmeyi unutmayın!